# Survival Analysis

## Overview

Survival analysis models time-to-event data, where the event may not have occurred for all subjects by the end of the study (censoring). Ignoring censoring by treating it as a non-event produces severely biased estimates.

**Core concepts:**

| Concept | Meaning |
|---|---|
| Survival function S(t) | P(T > t): probability of surviving past time t |
| Hazard function h(t) | Instantaneous event rate at t given survival to t |
| Censoring | Event not observed; we know only T > t_obs |
| Kaplan-Meier | Non-parametric S(t) estimator |
| Cox PH | Semi-parametric regression; proportional hazards |

**Ecological framing:** time until a restored site reaches target species richness (event), with some sites not yet achieving it by study end (right-censored).

**Python package:** `lifelines` — purpose-built for survival analysis with a clean API.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
n = 250
# Time-to-recovery for restored stream sites
treatment_intensity = rng.choice(['low','medium','high'], n,
                                  p=[0.3, 0.4, 0.3])
elevation = rng.uniform(50, 350, n)
nitrate   = rng.gamma(2, 2, n)
# True hazard: higher intensity = faster recovery
base_rate = {'low': 0.04, 'medium': 0.08, 'high': 0.14}
hazard    = np.array([base_rate[t] for t in treatment_intensity])
hazard   *= np.exp(-0.003*elevation + 0.05*nitrate)
true_time = rng.exponential(1/hazard)
study_end = 36  # months
observed  = true_time <= study_end
duration  = np.minimum(true_time, study_end)
df = pd.DataFrame(dict(duration=duration, event=observed.astype(int),
    treatment=treatment_intensity, elevation=elevation, nitrate=nitrate))
print(f"n={n}, events={observed.sum()} ({observed.mean():.0%}), censored={n-observed.sum()}")
print(df.groupby('treatment')['event'].agg(['sum','mean']).round(3))

---
## Kaplan-Meier Estimator

In [ ]:
try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import logrank_test
    fig, ax = plt.subplots(figsize=(9,5))
    colors = {'low':'steelblue','medium':'#e74c3c','high':'#4fffb0'}
    for group in ['low','medium','high']:
        mask = df['treatment'] == group
        kmf = KaplanMeierFitter()
        kmf.fit(df.loc[mask,'duration'], df.loc[mask,'event'],
                label=f'{group} intensity (n={mask.sum()})')
        kmf.plot_survival_function(ax=ax, color=colors[group], ci_show=True)
    ax.set_xlabel('Time (months)'); ax.set_ylabel('P(not yet recovered)')
    ax.set_title('Kaplan-Meier: Time to Recovery by Treatment Intensity')
    plt.tight_layout(); plt.show()
    # Log-rank test
    low  = df[df['treatment']=='low']
    high = df[df['treatment']=='high']
    lr = logrank_test(low['duration'], high['duration'],
                      low['event'], high['event'])
    print(f"Log-rank test (low vs high): p={lr.p_value:.4f}")
    print(f"  {'Significant difference in survival curves' if lr.p_value<0.05 else 'No significant difference'}")
except ImportError:
    print("pip install lifelines")
    print("KaplanMeierFitter().fit(duration, event).plot_survival_function()")

---
## Cox Proportional Hazards Model

In [ ]:
try:
    from lifelines import CoxPHFitter
    cox_df = df.copy()
    cox_df['trt_medium'] = (cox_df['treatment']=='medium').astype(int)
    cox_df['trt_high']   = (cox_df['treatment']=='high').astype(int)
    cox_df['elevation_s'] = (cox_df['elevation']-cox_df['elevation'].mean())/cox_df['elevation'].std()
    cox_df['nitrate_s']   = (cox_df['nitrate']  -cox_df['nitrate'].mean())  /cox_df['nitrate'].std()
    cph = CoxPHFitter()
    cph.fit(cox_df[['duration','event','trt_medium','trt_high',
                     'elevation_s','nitrate_s']],
            duration_col='duration', event_col='event')
    cph.print_summary()
    # Hazard ratios
    print("\nHazard Ratios (exp(coef)):")
    print(cph.hazard_ratios_.round(3))
except ImportError:
    print("pip install lifelines")

---
## Proportional Hazards Assumption

In [ ]:
try:
    from lifelines import CoxPHFitter
    from lifelines.statistics import proportional_hazard_test
    cox_df = df.copy()
    cox_df['trt_high']    = (cox_df['treatment']=='high').astype(int)
    cox_df['elevation_s'] = (cox_df['elevation']-cox_df['elevation'].mean())/cox_df['elevation'].std()
    cph2 = CoxPHFitter()
    cph2.fit(cox_df[['duration','event','trt_high','elevation_s']],
             duration_col='duration', event_col='event')
    # Schoenfeld residuals test
    ph_test = proportional_hazard_test(cph2, cox_df[['duration','event','trt_high','elevation_s']],
                                        time_transform='rank')
    print("Proportional Hazards test (Schoenfeld residuals):")
    print(ph_test.summary)
    print("\np > 0.05 for each covariate -> PH assumption holds")
    print("p < 0.05 -> time-varying effect -> consider stratification or time interaction")
    cph2.check_assumptions(cox_df[['duration','event','trt_high','elevation_s']],
                           show_plots=True)
    plt.tight_layout(); plt.show()
except ImportError:
    print("pip install lifelines")

In [ ]:
try:
    from lifelines import WeibullAFTFitter
    # Accelerated Failure Time model: parametric alternative to Cox
    aft_df = df.copy()
    aft_df['trt_high']    = (aft_df['treatment']=='high').astype(int)
    aft_df['elevation_s'] = (aft_df['elevation']-aft_df['elevation'].mean())/aft_df['elevation'].std()
    aft = WeibullAFTFitter()
    aft.fit(aft_df[['duration','event','trt_high','elevation_s']],
            duration_col='duration', event_col='event')
    aft.print_summary()
    print("\nAFT interpretation: positive coef -> longer time to event (slower recovery)")
    print("Cox interpretation: positive coef -> higher hazard (faster event)")
    print("Both are valid; choose based on whether accelerated time or hazard is more interpretable")
except ImportError:
    print("pip install lifelines")

---

## Common Pitfalls

**1. Ignoring censoring and analysing only observed events**  
Dropping censored observations (those that did not experience the event) or treating censoring time as a non-event time produces severely biased survival estimates — it underestimates survival time by excluding the longest-surviving units. Always use Kaplan-Meier or Cox models that properly account for censoring.

**2. Not testing the proportional hazards assumption**  
The Cox model assumes hazard ratios are constant over time. If a treatment has an early strong effect that fades, the PH assumption is violated and the Cox coefficient is an average that misrepresents both the early and late effects. Always run the Schoenfeld residuals test and plot scaled Schoenfeld residuals against time.

**3. Using the log-rank test when hazard functions cross**  
The log-rank test is most powerful when hazard functions are proportional. When survival curves cross (one group starts better, then worse), the log-rank test has low power. Use the Wilcoxon (Breslow) test or weighted log-rank tests when curves cross.

**4. Interpreting Cox coefficients as relative risks rather than hazard ratios**  
The exponentiated Cox coefficient is a hazard ratio — the ratio of instantaneous event rates. It is not a relative risk or an odds ratio, though it approximates both for rare events. Always label Cox results as hazard ratios.

**5. Not checking the informative censoring assumption**  
Survival analysis assumes censoring is uninformative — the reason a unit is censored is unrelated to its underlying survival time. If sites with worse water quality are more likely to drop out of the study, censoring is informative and standard estimators are biased. Always investigate the mechanism of censoring.
---
*python_methods_library - Samantha McGarrigle*